# MADelon Feature Selection
This notebook explores filter-based and sequential feature selection with Logistic Regression on the MADelon classification dataset. It compares a baseline with models trained on selected features, using cross-validation on the training split and a held-out test set for final evaluation.

> The notebook is an educational workflow. The README documents important evaluation caveats in the current implementation.

## Notebook Setup

The dependencies used throughout the analysis are collected in the next cell.

In [1]:
import numpy as np
import pandas as pd

from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from mlxtend.evaluate import bias_variance_decomp
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## 1. Dataset Overview and Loading

The CSV contains the target column `T` and the remaining columns are used as model features. The data file is named `mdlon.csv`.

In [2]:
df = pd.read_csv("mdlon.csv")

### Initial Data Checks

Preview the rows and check the table dimensions and missing-value status before separating the target from the features.

In [3]:
df.shape

(2000, 501)

In [4]:
df.isna().any().any()

np.False_

In [4]:
X = df.drop("T", axis=1)
y = df["T"]

## 2. Train-Test Split

The data is split once into training and test sets. Cross-validation and feature-selection work are performed using the training split; the test split is reserved for the final accuracy measurement.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

## 3. Baseline Logistic Regression

The baseline establishes a reference score before feature selection. Its cross-validation accuracy is calculated on the training set, then the fitted model is evaluated on the held-out test set.

In [ ]:
lr = LogisticRegression(max_iter=1000)

scores = cross_val_score(
	lr,
	X_train,
	y_train,
	cv = 5
)
print("Base line mean cv is: ",np.mean(scores))


In [36]:
lr.fit(X_train,y_train)
y_pred = lr.predict(X_test)
print('Baseline Accuracy is: ',accuracy_score(y_test,y_pred))

Baseline Accuracy is:  0.515


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 4. Filter-Based Feature Selection

Filter methods rank or remove features using properties of the data, before a predictive model is fitted. The notebook explores variance, pairwise correlation, and ANOVA F-scores.

### 4.1 Variance Threshold

Variance thresholding removes features whose values vary too little across the training samples. Low-variance features often carry little discriminative information, although the threshold should be interpreted in relation to the feature scales.

In [8]:
sel = VarianceThreshold(threshold=0.01)
sel.fit(X_train)

VarianceThreshold(threshold=0.01)

In [9]:
X_train_selected = sel.transform(X_train)
X_test_selected = sel.transform(X_test)

In [10]:
print("Original features:", X_train.shape[1])
print("Selected features:", X_train_selected.shape[1])

Original features: 500
Selected features: 500


In [11]:
sel.variances_

array([4.06693590e+01, 8.78226718e+02, 1.55035549e+03, 8.31291938e+01,
       1.73756374e+03, 4.61215590e+01, 1.63778246e+03, 1.96484375e+00,
       2.23556809e+02, 5.13596234e+01, 1.35122056e+03, 7.85132684e+02,
       1.19105934e+03, 1.79956065e+03, 1.47209694e+02, 4.82618137e+02,
       1.79105373e+02, 9.22065350e+02, 5.44050000e+02, 9.78244809e+01,
       1.60007344e+02, 5.03736059e+02, 4.73208609e+01, 9.31959059e+01,
       1.94124610e+03, 8.21492328e+02, 4.67169834e+02, 1.39064131e+02,
       2.10949600e+02, 4.87577890e+02, 5.94418121e+01, 1.33933893e+03,
       1.34061465e+03, 1.64827140e+03, 5.44824990e+02, 1.85544158e+03,
       1.32866921e+02, 5.14533686e+02, 2.19994496e+01, 2.56442148e+00,
       1.17809496e+01, 1.50348875e+03, 4.88242600e+02, 2.62230725e+02,
       7.97520625e+02, 5.81062923e+02, 1.26349624e+03, 1.05096311e+03,
       1.85162532e+03, 6.66127812e+02, 1.91734665e+03, 1.28645226e+03,
       5.13957100e+02, 2.62093246e+01, 8.30375600e+02, 4.31301653e+02,
      

All features were retained at a variance threshold of `0.01`, so this step removed no columns.

In [12]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

### 4.2 Correlation Filtering

Pairwise correlation is used to flag highly similar features. Here, pairs with correlation above `0.90` are considered redundant; the code computes this using the training split.

In [13]:
columns = X.columns

X_train_scaled_df = pd.DataFrame(X_train_scaled,columns=columns)
corr_matrix = X_train_scaled_df.corr()

In [14]:
X_test_scaled_df = pd.DataFrame(X_test_scaled,columns=columns)

In [15]:
columns_to_drop = []
columns = corr_matrix.columns

for i in range(len(columns)):
	for j in range(i+1,len(columns)):
		if corr_matrix.loc[columns[i],columns[j]] > 0.90:
			columns_to_drop.append(columns[i])

columns_to_drop = set(columns_to_drop)

In [16]:
X_test_scaled_df.drop(columns=columns_to_drop,inplace=True)

In [17]:
X_train_scaled_df.drop(columns=columns_to_drop,inplace=True)

In [18]:
# X_test_scaled_df.drop(columns=columns_to_drop,axis=1,inplace=True)

In [19]:
X_train_scaled_df.shape
X_test_scaled_df.shape

(400, 490)

Correlation filtering removed 10 features with pairwise correlation above `0.90`.

### 4.3 ANOVA F-Test with SelectKBest

The ANOVA F-test scores each feature by how strongly its values differ across target classes. `SelectKBest` keeps the 70 highest-scoring features; this is a univariate filter and does not capture interactions between features.

In [20]:
sel = SelectKBest(f_classif, k=70).fit(X_train_scaled_df, y_train)
X_train_scaled = sel.fit_transform(X_train_scaled_df, y_train)
X_test_scaled = sel.transform(X_test_scaled_df)

In [21]:
X_train_scaled.shape

(1600, 70)

The ANOVA selector retains the 70 highest-scoring features for the final model evaluation.

## 5. Cross-Validation After ANOVA Selection

This cell estimates accuracy after the ANOVA selector has been fitted on the full training split. Because feature selection happened before these folds were created, this score can be optimistic and should be treated as exploratory rather than as a fully leakage-free estimate.

In [22]:
lr = LogisticRegression(max_iter=1000)

scores = cross_val_score(
    lr,
    X_train_scaled,
    y_train,
    cv=5,
)

print(scores)
print(np.mean(scores))
print(np.std(scores))

[0.653125 0.634375 0.653125 0.634375 0.634375]
0.641875
0.009185586535436885


In the recorded comparison, `k = 70`, `100`, `200`, and `250` produced mean CV accuracies of `0.64`, `0.64`, `0.64`, and `0.62`. The notebook chose `k = 70` for a similar mean score with fewer features. Because these values were compared using the same training data, this choice is exploratory and can make subsequent CV estimates optimistic.

## 6. Sequential Feature Selection

Sequential Feature Selection (SFS) evaluates feature subsets using a Logistic Regression estimator. This notebook uses backward selection and chooses the subset size with the best internal cross-validation score.

In [23]:
lr = LogisticRegression()

sel = SFS(
	estimator=lr,
	forward=False,
	floating=False,
	cv=5,
	k_features='best',
	n_jobs=-1
)

model = sel.fit(X_train_scaled,y_train)


In [24]:
X_train_scaled_copy = X_train_scaled
X_test_scaled_copy = X_test_scaled

In [25]:
X_train_scaled = model.fit_transform(X_train_scaled,y_train)
X_test_scaled = model.transform(X_test_scaled)

In [26]:
scores = cross_val_score(
	lr,
	X_train_scaled,
	y_train,
	cv=5,
	n_jobs=-1
)
print(np.mean(scores))

0.683125


### Why the Initial SFS Score Was Leaky

Initially, SFS was fitted on the full training split before a separate cross-validation evaluation. The validation folds therefore influenced the feature subset chosen by SFS. Placing another SFS step inside the later cross-validation pipeline ensures that this step is fitted only on each outer fold's training portion.

The overall CV workflow still has leakage: variance filtering, scaling, correlation filtering, ANOVA selection, and the earlier SFS selection were all fitted on the full training split before the outer folds were created. The pipeline therefore evaluates folds using a feature set that was already influenced by all training rows.

In [27]:
lr = LogisticRegression()

model = lr.fit(
	X_train_scaled,
	y_train
)


In [28]:
y_predict = model.predict(X_test_scaled)

print('Accuracy Score: ',accuracy_score(y_test,y_predict))

Accuracy Score:  0.5575


The later SFS step is evaluated inside the cross-validation pipeline, so each fold refits that step using only its training portion. However, this pipeline receives data already reduced by variance, correlation, ANOVA, and an earlier SFS fit on the full training split; those earlier selections would also need to be inside the pipeline for a fully leakage-free CV workflow.

In [ ]:
lr = LogisticRegression(max_iter=1000)

sfs = SFS(
	estimator=lr,
	k_features='best',
	forward=False,
	floating=False,
	cv=5,
	n_jobs=-1
)

pipeline = Pipeline([
	('sfs',sfs),
	('model',lr)
])

cv = StratifiedKFold(
	n_splits=5,
	random_state=42,
	shuffle=True
)

scores = cross_val_score(
	estimator=pipeline,
	X=X_train_scaled,
	y=y_train,
	cv=cv,
	scoring='accuracy',
	n_jobs=-1
)

print(scores)
print(np.mean(scores))

[0.646875 0.6375   0.659375 0.6875   0.64375 ]
0.6549999999999999


The later SFS-in-pipeline cross-validation mean is approximately `65.5%` in the recorded run. That SFS step is refit within each outer fold, but the input data was already reduced using preprocessing and an earlier SFS fit on the full training split. Treat this as a development score, not a fully leakage-free estimate.

In [35]:
pipeline.fit(X_train_scaled,y_train)
y_pred = pipeline.predict(X_test_scaled)
print('Test accuracy: ',accuracy_score(y_test,y_pred))

Test accuracy:  0.5575


In [33]:
# X_train_scaled.shape
# X_test_scaled.shape
y_train.shape

(1600,)

## 7. Final Test Evaluation and Takeaway

The recorded baseline test accuracy is `51.5%`. The SFS pipeline's recorded test accuracy is `55.75%`. These test scores come from the held-out test split and are distinct from the cross-validation estimates.

In this single experiment, SFS + Logistic Regression achieved higher recorded accuracy than the baseline. This result is specific to this split and workflow, and it does not establish that SFS is generally superior. Because several preprocessing selectors are fitted before cross-validation, the CV estimate should be interpreted cautiously.